# 09 — Data Preparation for Toggle OOD Experiments (T1-T4, M1-M4)

These datasets support IID-vs-OOD comparison using the `toggle_ood` datasplit mechanism from the original CellOT paper.

The datasplit happens at **training time** via `config.yaml`, not in the data files themselves. This notebook creates the raw `.h5ad` files for 8 holdout groups:
- **T1-T4**: T cell holdout groups (CD8, CD8+thymocyte, all T subtypes, CD4)
- **M1-M4**: Monocyte holdout groups (non-classical, non-classical+generic, all monocyte subtypes, classical)

In [1]:
import sys
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")

import importlib
if "speciesot_helpers" in sys.modules:
    importlib.reload(sys.modules["speciesot_helpers"])

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse as sp_sparse
from speciesot_helpers import (
    top_n_organisms_from_species,
    match_cells_by_celltype_tissue,
    align_adatas_biomart_one2one,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("Imports OK")

/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Load Base Data

In [2]:
human_dir = '/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/'
mouse_dir = '/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/'

human_adatas = top_n_organisms_from_species(human_dir, -1, 'human')
mouse_adatas = top_n_organisms_from_species(mouse_dir, -1, 'mouse')

human_combined = ad.concat(human_adatas, join="outer")
mouse_combined = ad.concat(mouse_adatas, join="outer")

print(f"Human combined: {human_combined.shape}")
print(f"Mouse combined: {mouse_combined.shape}")

Human combined: (58852, 61759)
Mouse combined: (47802, 18024)


## 2. BioMart Alignment + HVG Selection + Cell Matching

In [3]:
print("Aligning genes via BioMart one-to-one orthologs ...")
mouse_all_aligned, human_all_aligned, ortholog_table = align_adatas_biomart_one2one(
    mouse_combined, human_combined
)

print(f"Ortholog pairs: {len(ortholog_table)}")
print(f"Mouse aligned: {mouse_all_aligned.shape}")
print(f"Human aligned: {human_all_aligned.shape}")

Aligning genes via BioMart one-to-one orthologs ...
Ortholog pairs: 14451
Mouse aligned: (47802, 14451)
Human aligned: (58852, 14451)


In [4]:
N_HVG = 1000

mouse_all_aligned.obs['condition'] = 'mouse'
human_all_aligned.obs['condition'] = 'human'

all_cells = ad.concat([mouse_all_aligned, human_all_aligned], join='inner')
print(f"Concatenated all cells: {all_cells.shape}")

sc.pp.highly_variable_genes(all_cells, n_top_genes=N_HVG, flavor='seurat')
hvg_genes = all_cells.var_names[all_cells.var.highly_variable].tolist()
print(f"Selected {len(hvg_genes)} HVGs")

mouse_all_hvg = mouse_all_aligned[:, hvg_genes].copy()
human_all_hvg = human_all_aligned[:, hvg_genes].copy()
mouse_all_hvg.obs['condition'] = 'mouse'
human_all_hvg.obs['condition'] = 'human'

print(f"Mouse HVG: {mouse_all_hvg.shape}")
print(f"Human HVG: {human_all_hvg.shape}")

Concatenated all cells: (106654, 14451)
Selected 1000 HVGs
Mouse HVG: (47802, 1000)
Human HVG: (58852, 1000)


In [5]:
mouse_matched_hvg, human_matched_hvg = match_cells_by_celltype_tissue(
    mouse_all_hvg, human_all_hvg
)

print(f"Matched mouse: {mouse_matched_hvg.shape}")
print(f"Matched human: {human_matched_hvg.shape}")

matched_hvg = ad.concat([mouse_matched_hvg, human_matched_hvg], join='inner')
print(f"\nTotal matched: {matched_hvg.shape}")
print(f"\nCell types in matched data:")
CT_COL = 'cell_type_ontology_term_id'
for ct_id, n in matched_hvg.obs[CT_COL].value_counts().items():
    label = ""
    if 'cell_type' in matched_hvg.obs.columns:
        match = matched_hvg.obs.loc[matched_hvg.obs[CT_COL] == ct_id, 'cell_type']
        if len(match) > 0:
            label = f" ({match.iloc[0]})"
    print(f"  {ct_id}{label}: {n} cells")

Matched mouse: (6418, 1000)
Matched human: (6418, 1000)

Total matched: (12836, 1000)

Cell types in matched data:
  CL:0008001 (hematopoietic precursor cell): 1584 cells
  CL:0000037 (hematopoietic stem cell): 1442 cells
  CL:0002393 (intermediate monocyte): 1008 cells
  CL:0000623 (natural killer cell): 912 cells
  CL:0000893 (thymocyte): 910 cells
  CL:0000875 (non-classical monocyte): 852 cells
  CL:1000320 (large intestine goblet cell): 760 cells
  CL:0002063 (pulmonary alveolar type 2 cell): 584 cells
  CL:0002543 (vein endothelial cell): 574 cells
  CL:0002548 (fibroblast of cardiac tissue): 484 cells
  CL:0002598 (bronchial smooth muscle cell): 440 cells
  CL:0000786 (plasma cell): 410 cells
  CL:0000236 (B cell): 406 cells
  CL:0000115 (endothelial cell): 406 cells
  CL:0000625 (CD8-positive, alpha-beta T cell): 390 cells
  CL:0000767 (basophil): 286 cells
  CL:0000084 (T cell): 204 cells
  CL:0000235 (macrophage): 204 cells
  CL:0000624 (CD4-positive, alpha-beta T cell): 190 

## 3. Group Definitions and Utilities

In [6]:
BASE_DIR = '/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT'
DATASET_DIR = os.path.join(BASE_DIR, 'cellot/cellot_gpu/datasets/speciesot-human-mouse')
CT_COL = 'cell_type_ontology_term_id'

keep_obs = ['condition', 'species', 'cell_type_ontology_term_id', 'cell_type',
            'tissue_ontology_term_id', 'tissue', 'donor_id']

def clean_adata(adata):
    """Strip layers/obsm/uns and densify X for CellOT compatibility."""
    obs_cols = [c for c in keep_obs if c in adata.obs.columns]
    X = adata.X
    if sp_sparse.issparse(X):
        X = np.array(X.todense())
    elif not isinstance(X, np.ndarray):
        X = np.array(X)
    obs = adata.obs[obs_cols].copy()
    for col in obs.columns:
        if hasattr(obs[col], 'cat') or str(obs[col].dtype) == 'category':
            obs[col] = obs[col].astype(str)
    return ad.AnnData(
        X=X.astype(np.float32),
        obs=obs,
        var=pd.DataFrame(index=adata.var_names),
    )


import h5py

def write_compat_h5ad(adata, path):
    """Write h5ad in old anndata format compatible with CellOT env (anndata <0.8).
    
    Bypasses anndata's writer to avoid format incompatibilities. Writes obs columns
    as plain string datasets, which old anndata reads as object-dtype columns.
    """
    with h5py.File(path, 'w') as f:
        X = adata.X
        if sp_sparse.issparse(X):
            X = np.array(X.todense())
        f.create_dataset('X', data=X.astype(np.float32))

        obs_grp = f.create_group('obs')
        index_vals = np.array(adata.obs.index.astype(str), dtype='S')
        obs_grp.create_dataset('_index', data=index_vals)
        obs_grp.attrs['_index'] = '_index'
        obs_grp.attrs['encoding-type'] = 'dataframe'
        obs_grp.attrs['encoding-version'] = '0.1.0'
        col_order = []
        for col in adata.obs.columns:
            vals = np.array(adata.obs[col].astype(str), dtype='S')
            obs_grp.create_dataset(col, data=vals)
            col_order.append(col)
        obs_grp.attrs['column-order'] = col_order

        var_grp = f.create_group('var')
        var_index = np.array(adata.var.index.astype(str), dtype='S')
        var_grp.create_dataset('_index', data=var_index)
        var_grp.attrs['_index'] = '_index'
        var_grp.attrs['encoding-type'] = 'dataframe'
        var_grp.attrs['encoding-version'] = '0.1.0'
        var_grp.attrs['column-order'] = []

GROUPS = {
    'T1': {
        'name': 'toggle_t1',
        'description': 'CD8 holdout',
        'holdout_ids': ['CL:0000625'],
        'holdout_label': 'cd8',
    },
    'T2': {
        'name': 'toggle_t2',
        'description': 'CD8 + thymocyte holdout',
        'holdout_ids': ['CL:0000625', 'CL:0000893'],
        'holdout_label': 'cd8_thymo',
    },
    'T3': {
        'name': 'toggle_t3',
        'description': 'All T cell subtypes holdout (CD4 + CD8 + thymocyte)',
        'holdout_ids': ['CL:0000624', 'CL:0000625', 'CL:0000893'],
        'holdout_label': 'tcell_subtype',
    },
    'T4': {
        'name': 'toggle_t4',
        'description': 'CD4 holdout',
        'holdout_ids': ['CL:0000624'],
        'holdout_label': 'cd4',
    },
    'M1': {
        'name': 'toggle_m1',
        'description': 'Non-classical monocyte holdout',
        'holdout_ids': ['CL:0000875'],
        'holdout_label': 'nonclassical_mono',
    },
    'M2': {
        'name': 'toggle_m2',
        'description': 'Non-classical + generic monocyte holdout',
        'holdout_ids': ['CL:0000875', 'CL:0000576'],
        'holdout_label': 'nonclassical_generic_mono',
    },
    'M3': {
        'name': 'toggle_m3',
        'description': 'All monocyte subtypes holdout',
        'holdout_ids': ['CL:0000875', 'CL:0000860', 'CL:0002393', 'CL:0000576'],
        'holdout_label': 'mono_subtype',
    },
    'M4': {
        'name': 'toggle_m4',
        'description': 'Classical monocyte holdout',
        'holdout_ids': ['CL:0000860'],
        'holdout_label': 'classical_mono',
    },
}

print(f"Groups defined: {list(GROUPS.keys())}")
for gid, g in GROUPS.items():
    print(f"  {gid}: {g['description']} \u2014 holdout {g['holdout_ids']}")

Groups defined: ['T1', 'T2', 'T3', 'T4', 'M1', 'M2', 'M3', 'M4']
  T1: CD8 holdout — holdout ['CL:0000625']
  T2: CD8 + thymocyte holdout — holdout ['CL:0000625', 'CL:0000893']
  T3: All T cell subtypes holdout (CD4 + CD8 + thymocyte) — holdout ['CL:0000624', 'CL:0000625', 'CL:0000893']
  T4: CD4 holdout — holdout ['CL:0000624']
  M1: Non-classical monocyte holdout — holdout ['CL:0000875']
  M2: Non-classical + generic monocyte holdout — holdout ['CL:0000875', 'CL:0000576']
  M3: All monocyte subtypes holdout — holdout ['CL:0000875', 'CL:0000860', 'CL:0002393', 'CL:0000576']
  M4: Classical monocyte holdout — holdout ['CL:0000860']


## 4. Generate Datasets for All Groups

In [7]:
os.makedirs(DATASET_DIR, exist_ok=True)

for gid, g in GROUPS.items():
    group_name = g['name']
    holdout_ids = set(g['holdout_ids'])
    label = g['holdout_label']

    print(f"\n{'=' * 70}")
    print(f"GROUP {gid}: {g['description']}")
    print(f"{'=' * 70}")

    # --- File 1: AE training data for OOD (holdout cells EXCLUDED) ---
    ae_all = ad.concat([mouse_all_hvg, human_all_hvg], join='inner')
    ae_mask = ~ae_all.obs[CT_COL].astype(str).isin(holdout_ids)
    ae_data_ood = clean_adata(ae_all[ae_mask].copy())

    ae_ood_path = os.path.join(DATASET_DIR, f'{group_name}_ae_training_ood_v07.h5ad')
    write_compat_h5ad(ae_data_ood, ae_ood_path)
    print(f"  AE training (OOD): {ae_data_ood.n_obs} cells -> {ae_ood_path}")

    # --- File 2: AE training data for IID (ALL cells, no exclusion) ---
    ae_data_iid = clean_adata(ae_all.copy())

    ae_iid_path = os.path.join(DATASET_DIR, f'{group_name}_ae_training_iid_v07.h5ad')
    write_compat_h5ad(ae_data_iid, ae_iid_path)
    print(f"  AE training (IID): {ae_data_iid.n_obs} cells -> {ae_iid_path}")

    # --- File 3: Matched dataset, unswapped (condition=species, for IMPACT) ---
    mouse_m = mouse_matched_hvg.copy()
    human_m = human_matched_hvg.copy()
    mouse_m.obs['condition'] = 'mouse'
    human_m.obs['condition'] = 'human'
    cellot_data = ad.concat([mouse_m, human_m], join='inner')
    cellot_data = clean_adata(cellot_data)

    cellot_path = os.path.join(DATASET_DIR, f'{group_name}_holdout_v07.h5ad')
    write_compat_h5ad(cellot_data, cellot_path)
    n_holdout = cellot_data.obs[CT_COL].astype(str).isin(holdout_ids).sum()
    print(f"  IMPACT: {cellot_data.n_obs} cells, {n_holdout} holdout -> {cellot_path}")

    # --- File 4: Matched dataset, swapped (condition=cell_type_status, for CellOT) ---
    mouse_s = mouse_matched_hvg.copy()
    human_s = human_matched_hvg.copy()
    swapped = ad.concat([mouse_s, human_s], join='inner')
    swapped.obs['species'] = swapped.obs['condition'].values
    is_holdout = swapped.obs[CT_COL].astype(str).isin(holdout_ids)
    swapped.obs['condition'] = np.where(is_holdout, label, f'non_{label}')
    swapped = clean_adata(swapped)

    swapped_path = os.path.join(DATASET_DIR, f'{group_name}_holdout_swapped_v07.h5ad')
    write_compat_h5ad(swapped, swapped_path)
    print(f"  CellOT: {swapped.n_obs} cells -> {swapped_path}")
    print(f"    condition values: {dict(swapped.obs['condition'].value_counts())}")

print("\n" + "=" * 70)
print("All datasets generated.")


GROUP T1: CD8 holdout


  AE training (OOD): 104656 cells -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/toggle_t1_ae_training_ood_v07.h5ad
  AE training (IID): 106654 cells -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/toggle_t1_ae_training_iid_v07.h5ad
  IMPACT: 12836 cells, 390 holdout -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/toggle_t1_holdout_v07.h5ad
  CellOT: 12836 cells -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/toggle_t1_holdout_swapped_v07.h5ad
    condition values: {'non_cd8': np.int64(12446), 'cd8': np.int64(390)}

GROUP T2: CD8 + thymocyte holdout
  AE training (OOD): 103201 cells -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/toggle_t2_ae_training_ood_v07.h5ad
  AE training (IID): 106654 cells -> /n/holylabs/mooney_lab/Lab/junyizh

## 5. Verify Generated Files

In [8]:
print("Generated dataset files:")
print("=" * 70)
for gid, g in GROUPS.items():
    group_name = g['name']
    files = [
        f'{group_name}_ae_training_ood_v07.h5ad',
        f'{group_name}_ae_training_iid_v07.h5ad',
        f'{group_name}_holdout_v07.h5ad',
        f'{group_name}_holdout_swapped_v07.h5ad',
    ]
    print(f"\nGroup {gid} ({g['description']}):")
    for f in files:
        path = os.path.join(DATASET_DIR, f)
        if os.path.exists(path):
            data = ad.read_h5ad(path)
            print(f"  {f}: {data.shape}")
        else:
            print(f"  {f}: NOT FOUND")

Generated dataset files:

Group T1 (CD8 holdout):
  toggle_t1_ae_training_ood_v07.h5ad: (104656, 1000)
  toggle_t1_ae_training_iid_v07.h5ad: (106654, 1000)
  toggle_t1_holdout_v07.h5ad: (12836, 1000)
  toggle_t1_holdout_swapped_v07.h5ad: (12836, 1000)

Group T2 (CD8 + thymocyte holdout):
  toggle_t2_ae_training_ood_v07.h5ad: (103201, 1000)
  toggle_t2_ae_training_iid_v07.h5ad: (106654, 1000)
  toggle_t2_holdout_v07.h5ad: (12836, 1000)
  toggle_t2_holdout_swapped_v07.h5ad: (12836, 1000)

Group T3 (All T cell subtypes holdout (CD4 + CD8 + thymocyte)):
  toggle_t3_ae_training_ood_v07.h5ad: (101202, 1000)
  toggle_t3_ae_training_iid_v07.h5ad: (106654, 1000)
  toggle_t3_holdout_v07.h5ad: (12836, 1000)
  toggle_t3_holdout_swapped_v07.h5ad: (12836, 1000)

Group T4 (CD4 holdout):
  toggle_t4_ae_training_ood_v07.h5ad: (104655, 1000)
  toggle_t4_ae_training_iid_v07.h5ad: (106654, 1000)
  toggle_t4_holdout_v07.h5ad: (12836, 1000)
  toggle_t4_holdout_swapped_v07.h5ad: (12836, 1000)

Group M1 (Non-